In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
术语替换器：直接返回结果；冲突自动降级；默认永不抛异常。

降级策略：
- 同源多目标        -> 取 priority 最高；平局保留首次
- 链式风险          -> 单遍替换，不二次处理
- 前缀/子串重叠     -> 最长优先
- 自映射            -> 跳过
- 空源词/非字符串   -> 跳过
- replace 异常      -> 返回原文
- 非 str 输入       -> str() 兜底
所有降级均记录到 self.warnings。
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple, Union


@dataclass(frozen=True)
class Term:
    source: str
    target: str
    priority: int = 0


class _TrieNode:
    __slots__ = ("children", "term")

    def __init__(self):
        self.children: Dict[str, "_TrieNode"] = {}
        self.term: Optional[Term] = None


class TermReplacer:
    def __init__(
        self,
        terms: Union[Dict[str, str], Iterable[Term]],
        *,
        case_sensitive: bool = True,
        ascii_boundary: bool = False,
        on_conflict: str = "resolve",   # "resolve" | "raise"
        detect_overlaps: bool = True,   # False 可跳过 O(n^2) 重叠检测
    ):
        if on_conflict not in {"resolve", "raise"}:
            raise ValueError(f"on_conflict 非法: {on_conflict}")
        self.case_sensitive = case_sensitive
        self.ascii_boundary = ascii_boundary
        self.on_conflict = on_conflict
        self.detect_overlaps = detect_overlaps

        self.root = _TrieNode()
        self.warnings: List[str] = []
        self._by_source: Dict[str, Term] = {}
        self._self_mapped: List[str] = []
        self._dup_conflicts: Dict[str, List[Tuple[str, int]]] = {}

        self._build(terms)

    # ============================ 构建 ============================
    def _build(self, terms) -> None:
        # 1) 规范化输入（dict / iterable / tuple / Term）
        raw = self._normalize_terms(terms)

        # 2) 按 source 分组，消解同源多目标
        grouped: Dict[str, List[Term]] = {}
        for t in raw:
            if t.source == t.target:
                if t.source not in self._self_mapped:
                    self._self_mapped.append(t.source)
                continue
            grouped.setdefault(t.source, []).append(t)

        for src, group in grouped.items():
            self._by_source[src] = self._resolve_group(src, group)

        # 3) 建 Trie（自映射已被过滤，不进入 Trie）
        for src, t in self._by_source.items():
            node = self.root
            for ch in src:
                k = self._key(ch)
                nxt = node.children.get(k)
                if nxt is None:
                    nxt = _TrieNode()
                    node.children[k] = nxt
                node = nxt
            node.term = t

        # 4) 链式风险检测（Trie 已就绪，用目标词扫一遍）
        for src, t in self._by_source.items():
            for _s, _e, hit in self._scan_raw(t.target):
                kind = "自包含" if hit.source == src else "含其他源词"
                self.warnings.append(
                    f"链式风险: {src!r} -> {t.target!r} 含 "
                    f"{hit.source!r} -> {hit.target!r}（{kind}），"
                    f"单遍替换不会二次处理"
                )

        # 5) 前缀 / 子串重叠检测（仅提示，最长优先自动消除）
        if self.detect_overlaps:
            sources = sorted(self._by_source.keys(), key=len)
            for i, short in enumerate(sources):
                for long_ in sources[i + 1:]:
                    if long_.startswith(short):
                        self.warnings.append(
                            f"前缀重叠: {short!r} 是 {long_!r} 的前缀，最长优先"
                        )
                    elif short in long_:
                        self.warnings.append(
                            f"子串重叠: {short!r} 是 {long_!r} 的非前缀子串，最长优先"
                        )

    def _normalize_terms(self, terms) -> List[Term]:
        out: List[Term] = []

        # dict 输入：key 天然唯一，不存在同源多目标
        if isinstance(terms, dict):
            for k, v in terms.items():
                if not isinstance(k, str) or not isinstance(v, str):
                    self.warnings.append(f"跳过非字符串术语: {k!r} -> {v!r}")
                    continue
                out.append(Term(k, v))
            return out

        # iterable 输入
        try:
            items = list(terms)
        except Exception as e:
            self.warnings.append(f"术语表无法迭代，已忽略: {e}")
            return out

        for item in items:
            t = self._coerce_term(item)
            if t is not None:
                out.append(t)
        return out

    def _coerce_term(self, item) -> Optional[Term]:
        if isinstance(item, Term):
            t = item
        elif isinstance(item, (tuple, list)) and len(item) >= 2:
            try:
                t = Term(
                    item[0],
                    item[1],
                    int(item[2]) if len(item) >= 3 else 0,
                )
            except Exception:
                self.warnings.append(f"跳过非法术语: {item!r}")
                return None
        else:
            self.warnings.append(f"跳过非法术语: {item!r}")
            return None

        if not isinstance(t.source, str) or not isinstance(t.target, str):
            self.warnings.append(f"跳过非字符串术语: {t!r}")
            return None
        if not t.source:
            self.warnings.append(f"跳过空源词: {t!r}")
            return None
        return t

    def _resolve_group(self, src: str, group: List[Term]) -> Term:
        targets = {g.target for g in group}
        if len(targets) == 1:
            return max(group, key=lambda x: x.priority)

        # 同源多目标 -> 记录并降级
        self._dup_conflicts[src] = [(g.target, g.priority) for g in group]
        detail = f"同源多目标冲突 {src!r} -> {self._dup_conflicts[src]}"

        if self.on_conflict == "raise":
            raise ValueError(f"{detail}（on_conflict='raise'）")

        best = group[0]
        for g in group[1:]:
            if g.priority > best.priority:
                best = g
        self.warnings.append(f"{detail}，降级选择 {best.target!r}")
        return best

    # ============================ 匹配 ============================
    def _key(self, ch: str) -> str:
        return ch if self.case_sensitive else ch.lower()

    @staticmethod
    def _is_ascii_word_char(ch: str) -> bool:
        return ch.isascii() and (ch.isalnum() or ch == "_")

    def _boundary_ok(self, text: str, start: int, end: int, term: Term) -> bool:
        if not self.ascii_boundary:
            return True
        if not term.source or not self._is_ascii_word_char(term.source[0]):
            return True
        if start > 0 and self._is_ascii_word_char(text[start - 1]):
            return False
        if end < len(text) and self._is_ascii_word_char(text[end]):
            return False
        return True

    def _match_at(self, text: str, i: int) -> Optional[Tuple[int, Term]]:
        node = self.root
        candidates: List[Tuple[int, Term]] = []
        j = i
        n = len(text)
        while j < n:
            node = node.children.get(self._key(text[j]))
            if node is None:
                break
            j += 1
            if node.term is not None:
                candidates.append((j, node.term))
        for end, term in reversed(candidates):
            if self._boundary_ok(text, i, end, term):
                return end, term
        return None

    def _scan_raw(self, text: str) -> List[Tuple[int, int, Term]]:
        result: List[Tuple[int, int, Term]] = []
        i = 0
        n = len(text)
        while i < n:
            m = self._match_at(text, i)
            if m is not None:
                end, term = m
                result.append((i, end, term))
                i = end
            else:
                i += 1
        return result

    # ============================ 对外接口 ============================
    def replace(self, text: str) -> str:
        """单遍替换。永不抛异常；任何异常返回原文。"""
        if not isinstance(text, str):
            try:
                text = str(text)
            except Exception as e:
                self.warnings.append(f"输入无法转 str，已原样返回: {e}")
                return text  # type: ignore[return-value]

        try:
            out: List[str] = []
            i = 0
            n = len(text)
            while i < n:
                m = self._match_at(text, i)
                if m is not None:
                    end, term = m
                    out.append(term.target)
                    i = end  # 单遍：跳过已匹配的原文
                else:
                    out.append(text[i])
                    i += 1
            return "".join(out)
        except Exception as e:
            self.warnings.append(f"替换异常，已返回原文: {e!r}")
            return text

    def scan(self, text: str) -> List[Tuple[int, int, str, str]]:
        """返回命中区间 (start, end, source, target)。异常时返回已收集结果。"""
        if not isinstance(text, str):
            try:
                text = str(text)
            except Exception:
                return []
        try:
            return [(s, e, t.source, t.target) for s, e, t in self._scan_raw(text)]
        except Exception as e:
            self.warnings.append(f"扫描异常: {e!r}")
            return []

    @property
    def active_terms(self) -> Dict[str, str]:
        """最终生效的术语映射。"""
        return {s: t.target for s, t in self._by_source.items()}

    def explain(self) -> str:
        lines: List[str] = ["【生效术语】"]
        for s, t in self.active_terms.items():
            lines.append(f"  {s!r} -> {t!r}")
        if self._self_mapped:
            lines.append("【自映射已跳过】")
            lines.extend(f"  {s!r}" for s in self._self_mapped)
        if self.warnings:
            lines.append(f"【警告 {len(self.warnings)} 条】")
            lines.extend(f"  - {w}" for w in self.warnings)
        else:
            lines.append("【无警告】")
        return "\n".join(lines)


# ============================ 便捷函数 ============================
def replace_terms(
    text: str,
    terms: Union[Dict[str, str], Iterable[Term]],
    **kwargs,
) -> str:
    """一行调用，永不抛异常。"""
    try:
        return TermReplacer(terms, **kwargs).replace(text)
    except Exception:
        return text if isinstance(text, str) else str(text)


# ============================ 测试 ============================
if __name__ == "__main__":
    print("=" * 70)
    print("场景 1：正常替换（含前缀重叠，最长优先）")
    print("=" * 70)
    terms = {
        "原料": "原材料",
        "基础材料": "原材料",
        "辅料": "原材料",
        "供货商": "供应商",
        "商家": "供应商",
        "卖家": "供应商",
        "原材料采购表": "原材料采购记录",
        "原材料采购单": "原材料采购记录",
        "原材料订单": "原材料采购记录",
        "原材料库存": "原材料库存",
        "原材料存货": "原材料库存",
        "原材料存量": "原材料库存",
        "产品订单": "客户订单",
    }
    r = TermReplacer(terms)
    for text in [
        "商家提供的原材料采购表和辅料库存，供货商确认产品订单",
        "原材料存货不足，基础材料需要补货",
        "原材料采购单已生成",
        "原材料库存充足",
        "卖家提交了原材料订单",
    ]:
        print(f"原文: {text}")
        print(f"结果: {r.replace(text)}")
        print(f"命中: {r.scan(text)}")
        print("-" * 70)
    print(r.explain())
    print()

    print("=" * 70)
    print("场景 2：同源多目标降级（取 priority 最高）")
    print("=" * 70)
    r2 = TermReplacer([
        Term("商家", "供应商", 10),
        Term("商家", "卖家", 5),
    ])
    print(f"replace('商家') = {r2.replace('商家')!r}")
    print(r2.explain())
    print()

    print("=" * 70)
    print("场景 3：同源多目标且 priority 相同（平局保留首次）")
    print("=" * 70)
    r3 = TermReplacer([
        Term("商家", "供应商"),
        Term("商家", "卖家"),
    ])
    print(f"replace('商家') = {r3.replace('商家')!r}")
    print(r3.explain())
    print()

    print("=" * 70)
    print("场景 4：链式风险（单遍替换，不二次处理）")
    print("=" * 70)
    r4 = TermReplacer({"商家": "供应商", "供应商": "供货方"})
    print(f"replace('商家') = {r4.replace('商家')!r}  （不会变成'供货方'）")
    print(r4.explain())
    print()

    print("=" * 70)
    print("场景 5：非法输入自动降级")
    print("=" * 70)
    r5 = TermReplacer([
        ("商家", "供应商"),
        ("", "空源词"),           # 空源词 -> 跳过
        ("卖家", 123),            # 非 str   -> 跳过
        "not a term",             # 非法条目 -> 跳过
        ("供货商", "供应商"),
    ])
    print(f"replace('商家和卖家由供货商提供') = "
          f"{r5.replace('商家和卖家由供货商提供')!r}")
    print(f"replace(12345) = {r5.replace(12345)!r}")
    print(r5.explain())
    print()

    print("=" * 70)
    print("场景 6：便捷函数（永不抛异常）")
    print("=" * 70)
    print(replace_terms(
        "商家提供的原材料采购表和辅料库存",
        terms,
    ))
    # 即便传入非法术语表也不抛
    print(replace_terms("原文", None))

场景 1：正常替换（含前缀重叠，最长优先）
原文: 商家提供的原材料采购表和辅料库存，供货商确认产品订单
结果: 供应商提供的原材料采购记录和原材料库存，供应商确认客户订单
命中: [(0, 2, '商家', '供应商'), (5, 11, '原材料采购表', '原材料采购记录'), (12, 14, '辅料', '原材料'), (17, 20, '供货商', '供应商'), (22, 26, '产品订单', '客户订单')]
----------------------------------------------------------------------
原文: 原材料存货不足，基础材料需要补货
结果: 原材料库存不足，原材料需要补货
命中: [(0, 5, '原材料存货', '原材料库存'), (8, 12, '基础材料', '原材料')]
----------------------------------------------------------------------
原文: 原材料采购单已生成
结果: 原材料采购记录已生成
命中: [(0, 6, '原材料采购单', '原材料采购记录')]
----------------------------------------------------------------------
原文: 原材料库存充足
结果: 原材料库存充足
命中: []
----------------------------------------------------------------------
原文: 卖家提交了原材料订单
结果: 供应商提交了原材料采购记录
命中: [(0, 2, '卖家', '供应商'), (5, 10, '原材料订单', '原材料采购记录')]
----------------------------------------------------------------------
【生效术语】
  '基础材料' -> '原材料'
  '辅料' -> '原材料'
  '供货商' -> '供应商'
  '商家' -> '供应商'
  '卖家' -> '供应商'
  '原材料采购表' -> '原材料采购记录'
  '原材料采购单' -> '原材料采购记录'
  '原材料订单' -> 

In [4]:
print("=" * 70)
print("场景 1：正常替换（含前缀重叠，最长优先）")
print("=" * 70)
terms = {
    "原料": "原材料",
    "基础材料": "原材料",
    "辅料": "原材料",
    "供货商": "供应商",
    "商家": "供应商",
    "卖家": "供应商",
    "原材料采购表": "原材料采购记录",
    "原材料采购单": "原材料采购记录",
    "原材料订单": "原材料采购记录",
    "原材料库存": "原材料库存",
    "原材料存货": "原材料库存",
    "原材料存量": "原材料库存",
    "产品订单": "客户订单",
}
r = TermReplacer(terms)


场景 1：正常替换（含前缀重叠，最长优先）


In [8]:
print(f"结果: {r.replace('原料订单和产品订单不够了')}")

结果: 原材料订单和客户订单不够了


In [ ]:
if __name__ == "__main__":
    print("=" * 70)
    print("场景 1：正常替换（含前缀重叠，最长优先）")
    print("=" * 70)
    terms = {
        "原料": "原材料",
        "基础材料": "原材料",
        "辅料": "原材料",
        "供货商": "供应商",
        "商家": "供应商",
        "卖家": "供应商",
        "原材料采购表": "原材料采购记录",
        "原材料采购单": "原材料采购记录",
        "原材料订单": "原材料采购记录",
        "原材料库存": "原材料库存",
        "原材料存货": "原材料库存",
        "原材料存量": "原材料库存",
        "产品订单": "客户订单",
    }
    r = TermReplacer(terms)
    for text in [
        "商家提供的原材料采购表和辅料库存，供货商确认产品订单",
        "原材料存货不足，基础材料需要补货",
        "原材料采购单已生成",
        "原材料库存充足",
        "卖家提交了原材料订单",
    ]:
        print(f"原文: {text}")
        print(f"结果: {r.replace(text)}")
        print(f"命中: {r.scan(text)}")
        print("-" * 70)
    print("冲突报告:")
    print(r.report)
    print()

    print("=" * 70)
    print("场景 2：同源多目标冲突（默认 error）")
    print("=" * 70)
    try:
        TermReplacer({"商家": "供应商", "商家": "卖家"})
    except ValueError as e:
        print("✓ 已正确抛出冲突：")
        print(e)
    print()

    print("=" * 70)
    print("场景 3：同源多目标冲突（priority 自动消除）")
    print("=" * 70)
    r2 = TermReplacer(
        [Term("商家", "供应商", 10), Term("商家", "卖家", 5)],
        on_duplicate="priority",
    )
    print(f"replace('商家') = {r2.replace('商家')!r}")  # 供应商
    print(f"生效映射: {r2.active_terms}")
    print(f"冲突报告:\n{r2.report}")
    print()

    print("=" * 70)
    print("场景 4：链式风险检测")
    print("=" * 70)
    r3 = TermReplacer({"商家": "供应商", "供应商": "供货方"})
    print(f"replace('商家') = {r3.replace('商家')!r}  （单遍，不会二次变成供货方）")
    print(f"链式风险报告:\n{r3.report}")
    print()

    print("=" * 70)
    print("场景 5：strict_chain=True 时链式风险直接报错")
    print("=" * 70)
    try:
        TermReplacer(
            {"商家": "供应商", "供应商": "供货方"},
            strict_chain=True,
        )
    except ValueError as e:
        print("✓ 已正确抛出链式风险：")
        print(e)
    print()

    print("=" * 70)
    print("场景 6：前缀重叠 + 子串重叠检测")
    print("=" * 70)
    r4 = TermReplacer({
        "原材料": "物料",
        "原材料采购表": "原材料采购记录",
        "材料": "物资",  # 子串（非前缀）
    })
    print(f"replace('原材料采购表和原材料') = "
          f"{r4.replace('原材料采购表和原材料')!r}")
    print(f"冲突报告:\n{r4.report}")